*(Original request: explain conditional edge routing — see `2.0-langgraph-core-concepts.ipynb` for that content. This notebook covers post-1.0 features.)*

# LangGraph Post-1.0 Features

This notebook surveys features added to LangGraph after the 1.0 release. Each section has a minimal runnable example.

In [ ]:
# %pip install langgraph langchain-openai

## The Command Primitive

Command combines state update + routing in a single return. Replaces the dict-return + separate conditional edge pattern.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal

class State(TypedDict):
    value: int
    route: str

def decide(state: State) -> Command[Literal["path_a", "path_b"]]:
    if state["value"] > 10:
        return Command(update={"route": "A"}, goto="path_a")
    return Command(update={"route": "B"}, goto="path_b")

def path_a(state: State): return {"route": "A done"}
def path_b(state: State): return {"route": "B done"}

g = StateGraph(State)
g.add_node("decide", decide)
g.add_node("path_a", path_a)
g.add_node("path_b", path_b)
g.add_edge(START, "decide")
g.add_edge("path_a", END)
g.add_edge("path_b", END)
graph = g.compile()
print(graph.invoke({"value": 15, "route": ""}))
print(graph.invoke({"value": 5, "route": ""}))

## Subgraphs

Subgraphs let you compose graphs inside graphs — the building block for multi-agent teams. Full coverage in `6.0-subgraphs.ipynb`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SubState(TypedDict):
    data: str

def process(state: SubState):
    return {"data": state["data"].upper()}

sub = StateGraph(SubState)
sub.add_node("process", process)
sub.add_edge(START, "process")
sub.add_edge("process", END)
subgraph = sub.compile()

class MainState(TypedDict):
    data: str

parent = StateGraph(MainState)
parent.add_node("sub", subgraph)   # subgraph as node
parent.add_edge(START, "sub")
parent.add_edge("sub", END)
main = parent.compile()
print(main.invoke({"data": "hello"}))

## Functional API

The Functional API (`@entrypoint`, `@task`) lets you write graphs as regular Python functions with decorators instead of explicit node/edge wiring. Released in LangGraph 0.3+; useful for simpler graphs.

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import MemorySaver

@task
def fetch_data(query: str) -> str:
    # In a real app, call a search API here
    return f"Results for: {query}"

@task
def summarize(data: str) -> str:
    return f"Summary: {data[:50]}..."

@entrypoint(checkpointer=MemorySaver())
def research_pipeline(query: str) -> str:
    data = fetch_data(query).result()
    summary = summarize(data).result()
    return summary

# Invoke like a regular graph
config = {"configurable": {"thread_id": "func-1"}}
result = research_pipeline.invoke("LangGraph features", config=config)
print(result)

## MCP Integration with `langchain-mcp-adapters`

The `langchain-mcp-adapters` package connects LangGraph agents to any Model Context Protocol (MCP) server — converting MCP tools to LangChain tools.

```bash
pip install langchain-mcp-adapters langgraph "langchain[openai]"
```

MCP servers expose tools via two transports:
- **stdio**: Local process (most common for local tools)
- **streamable HTTP with SSE fallback**: Remote servers

See: https://github.com/langchain-ai/langchain-mcp-adapters

In [ ]:
# MCP (Model Context Protocol) — connect an agent to a local MCP server via stdio
#
# my_mcp_server.py exposes three tools: get_weather, calculate, lookup_fact
# Run this cell as-is — the server is started automatically as a subprocess.

import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

client = MultiServerMCPClient(
    {
        "my_server": {
            "command": "python",
            "args": ["my_mcp_server.py"],
            "transport": "stdio",
        }
    }
)

tools = await client.get_tools()
print("Tools loaded from MCP server:", [t.name for t in tools])

model = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(model, tools)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What's the weather in Lisbon? Also, what is 42 * 7?"}]}
)
print(result["messages"][-1].content)


## Durable Execution

LangGraph Platform provides **durable execution** — if a run is interrupted (network error, timeout, server restart), it can be resumed from the last checkpoint. This is built on the same checkpointer system covered in `5.0-persistence-and-checkpointers.ipynb`. In production deployments, combine a SqliteSaver or PostgresSaver with the `langgraph deploy` command to get durable, resumable runs automatically.